In [1]:
from sys import base_prefix

import pandas as pd
from numpy.ma.extras import column_stack

#1.患者基本信息表(主表）
patients=pd.DataFrame({
    '患者ID':[1,2,3,4],
    '姓名':['张三','李四','王五','赵六'],
    '年龄':[45,67,52,38]
})

#2.就诊记录表（子表）
visits=pd.DataFrame({
    '患者ID':[1,2,2,3,5],
    '就诊日期':['2025-01-01','2025-01-05','2025-02-10','2025-01-20','2025-03-01'],
    '诊断':['肺炎','高血压','肺炎','糖尿病','骨折']
})

print("===患者表===")
print(patients)

print("\n===就诊表===")
print(visits)

#3.尝试4种合并方式，观察区别
print("\n=== 内连接 (Inner) - 只取交集 ===")
print(pd.merge(patients,visits,on='患者ID',how='inner'))

print("\n=== 左连接 (Left) - 保留所有患者 ===")
print(pd.merge(patients,visits,on="患者ID",how="left"))

print("\n=== 右连接 (Right) - 保留所有就诊记录 ===")
print(pd.merge(patients,visits,on='患者ID',how="right"))

print("\n=== 外连接 (Outer) - 全部保留 ===")
print(pd.merge(patients,visits,on='患者ID',how="outer"))

===患者表===
   患者ID  姓名  年龄
0     1  张三  45
1     2  李四  67
2     3  王五  52
3     4  赵六  38

===就诊表===
   患者ID        就诊日期   诊断
0     1  2025-01-01   肺炎
1     2  2025-01-05  高血压
2     2  2025-02-10   肺炎
3     3  2025-01-20  糖尿病
4     5  2025-03-01   骨折

=== 内连接 (Inner) - 只取交集 ===
   患者ID  姓名  年龄        就诊日期   诊断
0     1  张三  45  2025-01-01   肺炎
1     2  李四  67  2025-01-05  高血压
2     2  李四  67  2025-02-10   肺炎
3     3  王五  52  2025-01-20  糖尿病

=== 左连接 (Left) - 保留所有患者 ===
   患者ID  姓名  年龄        就诊日期   诊断
0     1  张三  45  2025-01-01   肺炎
1     2  李四  67  2025-01-05  高血压
2     2  李四  67  2025-02-10   肺炎
3     3  王五  52  2025-01-20  糖尿病
4     4  赵六  38         NaN  NaN

=== 右连接 (Right) - 保留所有就诊记录 ===
   患者ID   姓名    年龄        就诊日期   诊断
0     1   张三  45.0  2025-01-01   肺炎
1     2   李四  67.0  2025-01-05  高血压
2     2   李四  67.0  2025-02-10   肺炎
3     3   王五  52.0  2025-01-20  糖尿病
4     5  NaN   NaN  2025-03-01   骨折

=== 外连接 (Outer) - 全部保留 ===
   患者ID   姓名    年龄        就诊日期   诊断
0     1   张三  45.

In [18]:
#apply 是 Pandas 里极其常用的方法，专门用来对每一行或每一列应用一个自定义函数。它的核心价值在于：当内置的 mean/sum 等方法解决不了你的需求时，你需要用 apply 走“自定义逻辑”。

# 案例一：在医疗项目里，你经常需要把连续数值（年龄）切成几个档位（青年/中年/老年）
#定义一个分类函数
def age_group(age):
    if age<40:
        return '青年'
    elif age<60:
        return '中年'
    else:
        return '老年'

#用apply应用到年龄列
patients['年龄分组']=patients['年龄'].apply(age_group)
print(patients)

print()
# 案例二：如果你将来的医疗数据里有“诊断”文本，你需要从中提取特征（如是否包含“肺炎”一词），用 apply 非常适合。
def has_pneumonia(diagnosis):
    return 1 if '肺炎' in diagnosis else 0

visits['是否肺炎']=visits['诊断'].apply(has_pneumonia)
visits

   患者ID  姓名  年龄 年龄分组
0     1  张三  45   中年
1     2  李四  67   老年
2     3  王五  52   中年
3     4  赵六  38   青年



,患者ID,就诊日期,诊断,是否肺炎
0,1,2025-01-01,肺炎,1
1,2,2025-01-05,高血压,0
2,2,2025-02-10,肺炎,1
3,3,2025-01-20,糖尿病,0
4,5,2025-03-01,骨折,0


In [1]:
#pivot_table 的核心价值在于：把“长格式”数据变成“宽格式”特征表，方便你把多维度的临床指标整合成一行一行的患者特征，直接送入机器学习模型。


# 场景1：患者多次随访的血压记录（长格式)
bp_data=pd.DataFrame({
    '患者ID':[1,1,2,2,3],
    '随访时间':['第1次','第2次','第3次','第4次','第5次'],
    '收缩压':[120, 125, 140, 150, 130],
    '舒张压':[80, 82, 90, 95, 85]
})

print("长格式数据（每个患者有多行）：")
print(bp_data)

# 使用 pivot_table 转为宽格式（每个患者一行）
bp_wide=bp_data.pivot_table(
    index='患者ID',
    columns='随访时间',
    values=['收缩压','舒张压'],
    aggfunc='mean'
)

print("\n宽格式数据（每个患者一行，特征分散到各列）：")
print(bp_wide)

#肠镜2：患者多次就诊的检验指标（长格式）
lab_data = pd.DataFrame({
    '患者ID': [1, 1, 2, 2, 3, 3],
    '检验项目': ['血糖', '血脂', '血糖', '血脂', '血糖', '血脂'],
    '结果': [5.6, 4.2, 7.8, 5.1, 6.0, 4.8]
})

print("=== 原始检验数据（长格式） ===")
print(lab_data)

# 用 pivot_table 转成宽格式：每个患者一行，血糖和血脂作为两列
lab_pivot = lab_data.pivot_table(
    index='患者ID',
    columns='检验项目',
    values='结果',
    aggfunc='mean'
)

print("\n=== 转换后（每个患者一行，特征为检验项目） ===")
lab_pivot

NameError: name 'pd' is not defined

In [13]:
import pandas as pd

#1.患者基本信息表（主表）
patients=pd.DataFrame({
    '患者ID':[1,2,3,4],
    '姓名':['张三','李四','王五','赵六'],
    '年龄':[45,67,52,38],
    '手术类型':['胆囊切除', '心脏搭桥', '膝关节置换', '疝气修补']
})

#2.多次随访的检验记录（长格式）
labs=pd.DataFrame({
    '患者ID':[1,1,2,2,3,3,4,4],
    '随访时间':['术前', '术后', '术前', '术后1周', '术后1月', '术前', '术前', '术后'],
    '血红蛋白':[130, 110, 120, 100, 115, 125, 140, 120],
    '白细胞':[6.5, 12.0, 7.0, 15.0, 9.0, 6.0, 5.5, 10.0]
})

print("===患者主表===")
print(patients)
print("\n===检验记录（长格式）===")
print(labs)

# 3. 用 pivot_table 将检验记录转为宽格式（每个患者一行）
labs_wide=labs.pivot_table(
    index="患者ID",
    columns='随访时间',
    values=['血红蛋白','白细胞'],
    aggfunc='mean'    #----->重复时间点记录取平均值，看不懂可以问AI
)

# 展平多级列名（便于后续 merge）
labs_wide.columns = [f'{col[0]}_{col[1]}' for col in labs_wide.columns]
labs_wide = labs_wide.reset_index()

print('\n====检验记录转为宽格式====')
print(labs_wide)

# 4. 用 merge 将患者主表和检验宽表合并（左连接，保留所有患者）
merged=pd.merge(patients,labs_wide,on='患者ID',how='left')
print(merged)

# 5. 用 apply 生成风险等级标签（基于年龄和术后白细胞）
def risk_level(row):
    if pd.isna(row['白细胞_术后']):#为什么是pd.isna而不是像数据清洗时候的df.isna，涉及到apply的底层代码
        return '未知'
    if row['年龄']>60 and row['白细胞_术后']>12:
        return '高风险'
    elif row['年龄']>60 or row['白细胞_术后']>12:
        return '中风险'
    else:
        return '低风险'

merged['风险等级']=merged.apply(risk_level,axis=1)#axis是干嘛的？

print("\n=== 最终特征表（含风险等级标签） ===")
merged[['患者ID', '姓名', '年龄', '手术类型', '血红蛋白_术后', '白细胞_术后', '风险等级']]

===患者主表===
   患者ID  姓名  年龄   手术类型
0     1  张三  45   胆囊切除
1     2  李四  67   心脏搭桥
2     3  王五  52  膝关节置换
3     4  赵六  38   疝气修补

===检验记录（长格式）===
   患者ID  随访时间  血红蛋白   白细胞
0     1    术前   130   6.5
1     1    术后   110  12.0
2     2    术前   120   7.0
3     2  术后1周   100  15.0
4     3  术后1月   115   9.0
5     3    术前   125   6.0
6     4    术前   140   5.5
7     4    术后   120  10.0

====检验记录转为宽格式====
   患者ID  白细胞_术前  白细胞_术后  白细胞_术后1周  白细胞_术后1月  血红蛋白_术前  血红蛋白_术后  血红蛋白_术后1周  \
0     1     6.5    12.0       NaN       NaN    130.0    110.0        NaN   
1     2     7.0     NaN      15.0       NaN    120.0      NaN      100.0   
2     3     6.0     NaN       NaN       9.0    125.0      NaN        NaN   
3     4     5.5    10.0       NaN       NaN    140.0    120.0        NaN   

   血红蛋白_术后1月  
0        NaN  
1        NaN  
2      115.0  
3        NaN  
   患者ID  姓名  年龄   手术类型  白细胞_术前  白细胞_术后  白细胞_术后1周  白细胞_术后1月  血红蛋白_术前  血红蛋白_术后  \
0     1  张三  45   胆囊切除     6.5    12.0       NaN       NaN    130.0 

,患者ID,姓名,年龄,手术类型,血红蛋白_术后,白细胞_术后,风险等级
0,1,张三,45,胆囊切除,110.0,12.0,低风险
1,2,李四,67,心脏搭桥,NaN,NaN,未知
2,3,王五,52,膝关节置换,NaN,NaN,未知
3,4,赵六,38,疝气修补,120.0,10.0,低风险
